In [6]:
# src/model_training/train_spike_attention_masked.py
import torch
import torch.nn as nn
import torch.optim as optim
from torch import autocast, GradScaler
from tqdm import tqdm
import logging
import os
from datetime import datetime
import importlib

import src.model_training.model
importlib.reload(src.model_training.model)
from src.model_training.model import SpikeAttentionNet

In [7]:
def train_spike_attention(
    train_loader,
    val_loader,
    in_dim=1611,
    num_classes=7,
    embed_dim=256,
    num_heads=4,
    lr=1e-4,
    epochs=100,
    log_interval=50,
    save_path="checkpoints",
    use_bfloat16=True,
):
    # ---- Setup ----
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SpikeAttentionNet(
        in_dim=in_dim, embed_dim=embed_dim, num_heads=num_heads, num_classes=num_classes
    ).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scaler = GradScaler()
    dtype = torch.bfloat16 if use_bfloat16 else torch.float16

    # ---- Logging ----
    os.makedirs(save_path, exist_ok=True)
    log_file = os.path.join(save_path, f"train_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log")
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        handlers=[logging.FileHandler(log_file), logging.StreamHandler()],
    )

    logging.info(f"Using device: {device}")
    logging.info(f"Training SpikeAttentionNet with {sum(p.numel() for p in model.parameters()):,} parameters")

    best_val_acc = 0.0

    # ---- Epoch Loop ----
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, total_acc = 0.0, 0.0

        for batch, labels, mask in tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}"):
            batch, labels, mask = batch.to(device), labels.to(device), mask.to(device)

            optimizer.zero_grad(set_to_none=True)
            with autocast(device_type="cuda", dtype=dtype):
                logits = model(batch, mask=mask)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            preds = logits.argmax(dim=1)
            total_acc += (preds == labels).float().mean().item()
            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)
        avg_train_acc = total_acc / len(train_loader)

        # ---- Validation ----
        model.eval()
        val_loss, val_acc = 0.0, 0.0
        with torch.no_grad():
            for batch, labels, mask in val_loader:
                batch, labels, mask = batch.to(device), labels.to(device), mask.to(device)
                with autocast(device_type="cuda", dtype=dtype):
                    logits = model(batch, mask=mask)
                    loss = criterion(logits, labels)

                preds = logits.argmax(dim=1)
                val_acc += (preds == labels).float().mean().item()
                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)
        avg_val_acc = val_acc / len(val_loader)

        # ---- Logging ----
        logging.info(
            f"Epoch {epoch:03d} | "
            f"Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc:.4f} | "
            f"Val Loss: {avg_val_loss:.4f} | Val Acc: {avg_val_acc:.4f}"
        )
        
        os.makedirs(save_path, exist_ok=True)

        # ---- Save Best Model ----
        if avg_val_acc > best_val_acc:
            best_val_acc = avg_val_acc
            torch.save(model.state_dict(), os.path.join(save_path, "best_model.pt"))
            logging.info(f"✅ New best model saved at epoch {epoch} with val acc={best_val_acc:.4f}")

        # ---- Periodic Console Print ----
        if epoch % log_interval == 0 or epoch == 1:
            print(
                f"Epoch {epoch:03d} | "
                f"Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc:.4f} | "
                f"Val Loss: {avg_val_loss:.4f} | Val Acc: {avg_val_acc:.4f}"
            )

    logging.info(f"Training completed. Best validation accuracy: {best_val_acc:.4f}")
    return model


In [8]:
import src.load_dataset.dataset
importlib.reload(src.load_dataset.dataset)
from src.load_dataset.dataset import create_balanced_splits

train_loader, val_loader, test_loader = create_balanced_splits(
    features_path="data/features/audio_embeddings_feature_selection_emotion.pkl",
    labels_path="data/features/data_emotion.p",
    batch_size=64,
    T=25,
    augment=True
)

batch, labels, mask = next(iter(train_loader))
print("Batch shapes:", batch.shape, labels.shape, mask.shape)


✅ Loaded dataset: N=13708, F=1611
📊 Full class distribution: Counter({np.int64(0): 6436, np.int64(4): 2308, np.int64(1): 1636, np.int64(6): 1606, np.int64(3): 1003, np.int64(5): 361, np.int64(2): 358})
📁 Train: 10966 | Val: 1371 | Test: 1371
🧠 Train class distribution: Counter({np.int64(0): 5149, np.int64(4): 1846, np.int64(1): 1309, np.int64(6): 1285, np.int64(3): 802, np.int64(5): 289, np.int64(2): 286})
🧪 Val class distribution: Counter({np.int64(0): 643, np.int64(4): 231, np.int64(1): 164, np.int64(6): 161, np.int64(3): 100, np.int64(5): 36, np.int64(2): 36})
🧩 Test class distribution: Counter({np.int64(0): 644, np.int64(4): 231, np.int64(1): 163, np.int64(6): 160, np.int64(3): 101, np.int64(5): 36, np.int64(2): 36})
📈 Oversampling applied to train loader.
Batch shapes: torch.Size([64, 25, 1611]) torch.Size([64]) torch.Size([64, 25])


In [9]:
# Done: Pass to your training function
model = train_spike_attention(
    train_loader=train_loader,
    val_loader=val_loader,
    in_dim=1611,
    num_classes=7,
    epochs=100,
)


2025-10-23 14:47:12,909 | INFO | Using device: cuda
2025-10-23 14:47:12,910 | INFO | Training SpikeAttentionNet with 678,663 parameters
Epoch 1/100: 100%|██████████| 171/171 [00:16<00:00, 10.59it/s]
2025-10-23 14:47:29,653 | INFO | Epoch 001 | Train Loss: 1.9613 | Train Acc: 0.1490 | Val Loss: 1.8742 | Val Acc: 0.2601
2025-10-23 14:47:29,660 | INFO | ✅ New best model saved at epoch 1 with val acc=0.2601


Epoch 001 | Train Loss: 1.9613 | Train Acc: 0.1490 | Val Loss: 1.8742 | Val Acc: 0.2601


Epoch 2/100: 100%|██████████| 171/171 [00:16<00:00, 10.65it/s]
2025-10-23 14:47:46,336 | INFO | Epoch 002 | Train Loss: 1.9523 | Train Acc: 0.1498 | Val Loss: 1.8778 | Val Acc: 0.2850
2025-10-23 14:47:46,339 | INFO | ✅ New best model saved at epoch 2 with val acc=0.2850
Epoch 3/100: 100%|██████████| 171/171 [00:16<00:00, 10.65it/s]
2025-10-23 14:48:02,989 | INFO | Epoch 003 | Train Loss: 1.9506 | Train Acc: 0.1517 | Val Loss: 1.9157 | Val Acc: 0.2201
Epoch 4/100: 100%|██████████| 171/171 [00:17<00:00,  9.69it/s]
2025-10-23 14:48:21,706 | INFO | Epoch 004 | Train Loss: 1.9436 | Train Acc: 0.1690 | Val Loss: 1.9189 | Val Acc: 0.1632
Epoch 5/100: 100%|██████████| 171/171 [00:27<00:00,  6.20it/s]
2025-10-23 14:48:50,346 | INFO | Epoch 005 | Train Loss: 1.9382 | Train Acc: 0.1732 | Val Loss: 1.8937 | Val Acc: 0.1786
Epoch 6/100: 100%|██████████| 171/171 [00:27<00:00,  6.31it/s]
2025-10-23 14:49:18,501 | INFO | Epoch 006 | Train Loss: 1.9323 | Train Acc: 0.1852 | Val Loss: 1.9866 | Val Acc: 

Epoch 050 | Train Loss: 1.8613 | Train Acc: 0.2523 | Val Loss: 1.9277 | Val Acc: 0.1427


Epoch 51/100: 100%|██████████| 171/171 [00:15<00:00, 10.79it/s]
2025-10-23 15:01:42,111 | INFO | Epoch 051 | Train Loss: 1.8657 | Train Acc: 0.2449 | Val Loss: 1.9104 | Val Acc: 0.1530
Epoch 52/100: 100%|██████████| 171/171 [00:15<00:00, 10.78it/s]
2025-10-23 15:01:58,567 | INFO | Epoch 052 | Train Loss: 1.8650 | Train Acc: 0.2440 | Val Loss: 1.9434 | Val Acc: 0.1509
Epoch 53/100: 100%|██████████| 171/171 [00:15<00:00, 10.87it/s]
2025-10-23 15:02:14,897 | INFO | Epoch 053 | Train Loss: 1.8633 | Train Acc: 0.2455 | Val Loss: 1.8990 | Val Acc: 0.1722
Epoch 54/100: 100%|██████████| 171/171 [00:15<00:00, 10.76it/s]
2025-10-23 15:02:31,397 | INFO | Epoch 054 | Train Loss: 1.8617 | Train Acc: 0.2413 | Val Loss: 1.9072 | Val Acc: 0.1623
Epoch 55/100: 100%|██████████| 171/171 [00:15<00:00, 10.85it/s]
2025-10-23 15:02:47,766 | INFO | Epoch 055 | Train Loss: 1.8682 | Train Acc: 0.2385 | Val Loss: 1.9176 | Val Acc: 0.1521
Epoch 56/100: 100%|██████████| 171/171 [00:18<00:00,  9.11it/s]
2025-10-23 

Epoch 100 | Train Loss: 1.8591 | Train Acc: 0.2474 | Val Loss: 1.9105 | Val Acc: 0.1720


In [10]:
# Save only model weights (state_dict)
torch.save(model.state_dict(), "best_model.pth")